**План проекта**

В проекте вы реализуете pretrain и posttrain этапы обучения LLM. Выполняйте проект в Jupyter Notebook на ВМ. Выполнение заданий проекта займёт от 5 до 8 часов, не считая времени на обучение. 

**Pretrain**


Претрейн — самый ресурсоёмкий этап обучения LLM. Чтобы полноценно обучить даже небольшую модель (менее 1B), понадобится более 10к GPU-часов на A100. Чтобы не тратить недели на обучение, но отработать ключевые приёмы, в проекте вы выполните упрощённую задачу. 
При полноценном претрейне модель учится обобщать знания из данных, на которых происходило обучение, чтобы потом извлекать эти знания по текстовым запросам уже после обучения. Упростим задачу — научим модель только структуре языка. 
Сосредоточимся на одном узком домене — текстах произведений русской литературы — и обучим модель продолжать фразы из этого домена разумным текстом. 

In [81]:
from pathlib import Path
import pandas as pd
import re
from razdel import sentenize

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

from datasets import Dataset

from transformers import TrainerCallback, TrainingArguments, Trainer
import torch
from torch.nn.utils.rnn import pad_sequence

import gc

corpus_path = "corpus"
dataset_name = "russian_novels_dataset.parquet"
clean_dataset_name = "clean_" + dataset_name
chunks_dataset_name = "chunks_" + dataset_name

context_length = 512

Скачайте данные из репозитория и упакуйте их в один датасет. Вам понадобятся все произведения из репозитория.

In [82]:
corpus_path = Path(corpus_path)

data = []

for file in corpus_path.rglob("*.txt"):
    text = file.read_text(encoding="utf-8")
    
    data.append({
        "filename": file.name,
        "text": text
    })

df = pd.DataFrame(data)
df.to_parquet(dataset_name, index=False)

print(f"Количество произведений: {len(df)}")
print(f"Размер датасета: {df.shape}")
df.head()

Количество произведений: 108
Размер датасета: (108, 2)


,filename,text
0,Bulgakov_BelayaGvardiya.txt,\n Посвящается...
1,Bulgakov_Diavoliada.txt,"Повесть о том, как близнецы погуб..."
2,Bulgakov_Master.txt,Мастер И Маргарита\n\nМихаил Афанасиевич Булга...
3,Bulgakov_RokovyeYaytsa.txt,Глава 1. Куррикулюм витэ профессора Персикова\...
4,Bulgakov_TeatralnyjRoman.txt,Михаил Булгаков. Театральный роман\n* (Записки...


Проведите препроцессинг данных:

- Очистите их от дубликатов.
- Очистите от предложений с буквами не из кириллицы.
- Обработайте повторяющуюся пунктуацию и т. д.
- Разбейте на чанки поменьше, чтобы можно было добавить <bos> и <eos> токены в соответствии с обучаемой длиной контекста.

In [83]:
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Количество произведений после удаления дубликатов: {len(df)}")

Количество произведений после удаления дубликатов: 107


In [84]:
def split_sentences(text):
    return [sentence.text for sentence in sentenize(text)]

In [85]:
LATIN = re.compile(r"[A-Za-z]")

def is_valid_sentence(sentence):
    sentence = sentence.strip()
    
    if not sentence:
        return False
    
    if LATIN.search(sentence):
        return False
    
    if not re.search(r"[А-Яа-яЁё]", sentence):
        return False
    
    return True

In [86]:
def normalize_punctuation(text):
    text = re.sub(r"\?{2,}", "?", text)
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\.{4,}", "...", text)
    text = re.sub(r"-{2,}", "—", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [87]:
def clean_text(text):
    sentences = split_sentences(text)
    
    cleaned_sentences = []
    
    for sentence in sentences:
        sentence = normalize_punctuation(sentence)
        
        if not is_valid_sentence(sentence):
            continue
        
        cleaned_sentences.append(sentence)
    
    return " ".join(cleaned_sentences)

df["text"] = df["text"].apply(clean_text)
df.to_parquet(clean_dataset_name, index=False)

In [88]:
def make_text_chunks(text, max_chars=2000):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + max_chars
        
        if end < len(text):
            boundary = text.rfind(". ", start, end)
            
            if boundary > start:
                end = boundary + 1
        
        chunks.append(text[start:end].strip())
        start = end
    
    return chunks

In [89]:
chunks = []

for _, row in df.iterrows():
    text_chunks = make_text_chunks(row["text"])
    
    for chunk_id, chunk in enumerate(text_chunks):
        chunks.append({
            "filename": row["filename"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunks)
chunks_df.to_parquet(
    chunks_dataset_name,
    index=False
)

print(f"Количество чанков: {len(chunks_df)}")
chunks_df.head()

Количество чанков: 21857


,filename,chunk_id,text
0,Bulgakov_BelayaGvardiya.txt,0,Посвящается Любови Евгеньевне Белозерской Поше...
1,Bulgakov_BelayaGvardiya.txt,1,За что такая обида? Несправедливость? Зачем по...
2,Bulgakov_BelayaGvardiya.txt,2,Восемнадцатый год летит к концу и день ото дня...
3,Bulgakov_BelayaGvardiya.txt,3,"Гору замело, засыпало сарайчики во дворе - и с..."
4,Bulgakov_BelayaGvardiya.txt,4,"Баркаролла."" ""Недаром помнит вся Россия Про де..."


Создайте и обучите собственный токенизатор на полученных данных. Размер словаря выберите небольшим: при обучении только на рассмотренных текстах — около 3к токенов. В рассматриваемых данных язык намного менее разнообразен, чем в совокупных данных, поэтому крупные токенизаторы от реальных LLM могут не подойти.

In [90]:
special_tokens = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
]

In [91]:
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size=3000,
    min_frequency=2,
    special_tokens=special_tokens
)

def text_iterator():
    for text in chunks_df["text"]:
        yield text

tokenizer.train_from_iterator(
    text_iterator(),
    trainer=trainer
)

print("Размер словаря:", tokenizer.get_vocab_size())

for token in special_tokens:
    print(f"{token}: {tokenizer.token_to_id(token)}")


Размер словаря: 3000
<PAD>: 0
<UNK>: 1
<BOS>: 2
<EOS>: 3


In [92]:
text = chunks_df.iloc[0]["text"]

output = tokenizer.encode(text)

print(output.tokens[:100])
print(output.ids[:100])

print("Символов:", len(text))
print("Токенов:", len(output.ids))

unk_id = tokenizer.token_to_id("<UNK>")
print("\nКоличество <UNK>:", output.ids.count(unk_id))

decoded = tokenizer.decode(output.ids)

print("\nИсходный:")
print(text[:500])

print("\nПосле decode:")
print(decoded[:500])

['ÐŁ', 'Ð¾Ñģ', 'Ð²', 'ÑıÑī', 'Ð°ÐµÑĤÑģÑı', 'ĠÐĽÑİÐ±', 'Ð¾Ð²', 'Ð¸', 'ĠÐķÐ²', 'Ð³', 'ÐµÐ½ÑĮ', 'ÐµÐ²', 'Ð½Ðµ', 'ĠÐĳ', 'ÐµÐ»', 'Ð¾Ð·', 'ÐµÑĢ', 'ÑģÐºÐ¾Ð¹', 'ĠÐŁ', 'Ð¾ÑĪÐµÐ»', 'ĠÐ¼ÐµÐ»', 'ÐºÐ¸Ð¹', 'ĠÑģÐ½ÐµÐ³', 'ĠÐ¸', 'ĠÐ²Ð´ÑĢÑĥÐ³', 'ĠÐ¿Ð¾Ð²', 'Ð°Ð»', 'Ð¸Ð»', 'ĠÑħÐ»Ð¾Ð¿', 'ÑĮÑı', 'Ð¼Ð¸', '.', 'ĠÐĴ', 'ÐµÑĤ', 'ÐµÑĢ', 'ĠÐ·Ð°Ð²', 'ÑĭÐ»', ';', 'ĠÑģÐ´ÐµÐ»', 'Ð°Ð»Ð°ÑģÑĮ', 'ĠÐ¼ÐµÑĤ', 'ÐµÐ»ÑĮ', '.', 'ĠÐĴ', 'ĠÐ¾Ð´Ð½Ð¾', 'ĠÐ¼Ð³Ð½Ð¾Ð²', 'ÐµÐ½Ð¸Ðµ', 'ĠÑĤÐµÐ¼', 'Ð½Ð¾Ðµ', 'ĠÐ½ÐµÐ±', 'Ð¾', 'ĠÑģÐ¼ÐµÑĪ', 'Ð°Ð»Ð¾ÑģÑĮ', 'ĠÑģ', 'ĠÑģÐ½', 'ÐµÐ¶', 'Ð½ÑĭÐ¼', 'ĠÐ¼Ð¾ÑĢ', 'ÐµÐ¼', '.', 'ĠÐĴÑģÐµ', 'ĠÐ¸ÑģÑĩÐµÐ·', 'Ð»Ð¾', '.', 'Ġ-', 'ĠÐĿÑĥ', ',', 'ĠÐ±Ð°ÑĢ', 'Ð¸Ð½', ',', 'Ġ-', 'ĠÐ·Ð°Ðº', 'ÑĢÐ¸ÑĩÐ°Ð»', 'ĠÑı', 'Ð¼', 'ÑīÐ¸Ðº', ',', 'Ġ-', 'ĠÐ±ÐµÐ´', 'Ð°', ':', 'ĠÐ±ÑĥÑĢ', 'Ð°Ð½', '!', 'Ġ"', 'Ðļ', 'Ð°Ð¿', 'Ð¸ÑĤÐ°Ð½', 'ÑģÐºÐ°Ñı', 'ĠÐ´Ð¾Ñĩ', 'ÐºÐ°', '"', 'ĠÐĺ', 'ĠÑģÑĥÐ´', 'Ð¸Ð¼', 'Ñĭ', 'ĠÐ±ÑĭÐ»Ð¸', 'ĠÐ¼ÐµÑĢÑĤÐ²', 'ÑĭÐµ', 'ĠÐ¿Ð¾']
[1035, 190, 141, 786, 822, 2129, 149, 123, 2030, 236, 313, 230, 301, 398, 160, 278, 158, 842, 241, 957,

Токенизируйте данные и подготовьте их к претрейну с длиной контекста 512 токенов в виде экземпляра класса transformers.Dataset

In [93]:
bos_id = tokenizer.token_to_id("<BOS>")
eos_id = tokenizer.token_to_id("<EOS>")
pad_id = tokenizer.token_to_id("<PAD>")

In [94]:
def tokenize_and_chunk(text, tokenizer, context_length=512):
    token_ids = tokenizer.encode(text).ids
    
    chunk_size = context_length - 2
    
    chunks = []
    
    for i in range(0, len(token_ids), chunk_size):
        chunk = token_ids[i:i + chunk_size]
        
        if not chunk:
            continue
        
        chunk = [bos_id] + chunk + [eos_id]
        chunks.append(chunk)
    
    return chunks

In [95]:
tokenized_chunks = []

for text in chunks_df["text"]:
    tokenized_chunks.extend(
        tokenize_and_chunk(
            text,
            tokenizer,
            context_length=context_length
        )
    )

print("Количество последовательностей:", len(tokenized_chunks))

Количество последовательностей: 43265


In [96]:
lengths = [len(x) for x in tokenized_chunks]

print("Минимальная длина:", min(lengths))
print("Максимальная длина:", max(lengths))

Минимальная длина: 3
Максимальная длина: 512


In [97]:
dataset = Dataset.from_dict({
    "input_ids": tokenized_chunks
})

print("Первая последовательность:")
print(dataset[0]["input_ids"][:20])

print("\nBOS:", dataset[0]["input_ids"][0])
print("EOS:", dataset[0]["input_ids"][-1])
print("Длина:", len(dataset[0]["input_ids"]))

Первая последовательность:
[2, 1035, 190, 141, 786, 822, 2129, 149, 123, 2030, 236, 313, 230, 301, 398, 160, 278, 158, 842, 241]

BOS: 2
EOS: 3
Длина: 512


Инициализируйте модель ~150M параметров c произвольной decoder-only архитектурой трансформера. 

Например, можно рассмотреть LlamaConfig с параметрами hidden_size=1024, intermediate_size=1536, num_hidden_layers=16, num_attention_heads=16, num_key_value_heads=8.

In [98]:
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=tokenizer.get_vocab_size(),
    hidden_size=1024,
    intermediate_size=1902,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    max_position_embeddings=context_length,
    bos_token_id=bos_id,
    eos_token_id=eos_id,
    pad_token_id=pad_id,
)

model = LlamaForCausalLM(config)

In [99]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Количество параметров: {num_params}")

print("\nРазмер словаря:", config.vocab_size)
print("Размер контекста:", config.max_position_embeddings)
print("Hidden size:", config.hidden_size)
print("Intermediate size:", config.intermediate_size)
print("Количество слоёв:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)

Количество параметров: 149996544

Размер словаря: 3000
Размер контекста: 512
Hidden size: 1024
Intermediate size: 1902
Количество слоёв: 16
Attention heads: 16
KV heads: 8


Чтобы оценить качество, используйте промпты:
```
test_prompts = [
       "Все мысли, которые имеют огромные последствия",
       "Сила войска зависит от его духа",
       "Мысль о том, что он принес страдания",
       "Человек сознает себя свободным",
       "Что бы ни случилось, я всегда буду",
       "Любовь мешает смерти",
       "Нет, жизнь не кончена",
       "Всякая мысль, даже самая простая",
       "Война не любезность, а самое гадкое дело",
       "Чтобы жить честно"
   ]
```
    
Подготовьте коллбэки для валидации качества на промптах. Реализуйте обучение с помощью Trainer. Обратите внимание на параметры регуляризации weight_decay. Используйте подходящий batch_size — в диапазоне 64—128.

In [100]:
def collate_fn(features):
    sequences = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in features
    ]

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=pad_id
    )

    attention_mask = (input_ids != pad_id).long()

    labels = input_ids.clone()
    labels[input_ids == pad_id] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [101]:
lengths = [len(x["input_ids"]) for x in dataset]

print("Средняя длина:", sum(lengths) / len(lengths))
print("Медиана:", sorted(lengths)[len(lengths) // 2])
print("Максимум:", max(lengths))
print("Минимум:", min(lengths))

print(
    "Доля последовательностей длиной 512:",
    sum(x == 512 for x in lengths) / len(lengths)
)

Средняя длина: 325.2130590546631
Медиана: 471
Максимум: 512
Минимум: 3
Доля последовательностей длиной 512: 0.4949035016757194


In [102]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

In [103]:
class GenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        model.eval()

        device = torch.device("cuda")

        print("\n" + "=" * 80)
        print(f"Генерация после эпохи {state.epoch:.0f}")
        print("=" * 80)

        with torch.no_grad():
            for prompt in self.prompts:
                input_ids = self.tokenizer.encode(prompt).ids

                input_ids = torch.tensor(
                    [input_ids],
                    dtype=torch.long,
                    device=device
                )

                output_ids = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.95,
                    pad_token_id=pad_id,
                    eos_token_id=eos_id,
                )

                generated_text = self.tokenizer.decode(
                    output_ids[0].tolist()
                )

                print(f"\nPROMPT: {prompt}")
                print(f"OUTPUT: {generated_text}")

        model.train()

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama_russian_pretrain",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=16,
    gradient_checkpointing=True,

    num_train_epochs=5,

    learning_rate=3e-4,
    weight_decay=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=10,

    bf16=True,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    optim="adamw_torch_fused",

    report_to="none",
)

In [105]:
dataset_split = dataset.train_test_split(
    test_size=0.01,
    seed=42
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))

Train: 42832
Eval: 433


In [106]:
del trainer
gc.collect()
torch.cuda.empty_cache()

print(
    f"CUDA allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

CUDA allocated: 2.09 GB


In [107]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    callbacks=[
        GenerationCallback(
            tokenizer=tokenizer,
            prompts=test_prompts,
            max_new_tokens=50,
        )
    ],
)

In [ ]:
trainer.train()

trainer.save_model("./llama_russian_pretrain/final")

Epoch,Training Loss,Validation Loss
1,3.719669,3.730585



Генерация после эпохи 1

PROMPT: Все мысли, которые имеют огромные последствия
OUTPUT: Все мысли, которые имеют огромные последствия натуральных, которых не умели и не могло быть чувствоваться на то, что не то, чего она не замечает, и никогда нет как не предвидела, кроме чего она не могла не понимать, не

PROMPT: Сила войска зависит от его духа
OUTPUT: Сила войска зависит от его духа, певнами, выстраивавшими по литературному колхозу, в пеплах, и в углу, и в черных пушек, в красном кожи,

PROMPT: Мысль о том, что он принес страдания
OUTPUT: Мысль о том, что он принес страдания, и натуру, и в то же время он, и что ему надо было и потому не от него, а к тому, и не видал его и не могли сделать того и в том, что он сам не знает и что

PROMPT: Человек сознает себя свободным
OUTPUT: Человек сознает себя свободным с нимогрытыми куметиневесною для насрящейся статиныцского отцатия, а он, и ему не понимает, что ли, и наивно подаваемый бантик

PROMPT: Что бы ни случилось, я всегда буду
OUTPUT: Чт

TrainOutput(global_step=670, training_loss=4.562057102260305, metrics={'train_runtime': 1172.6631, 'train_samples_per_second': 36.525, 'train_steps_per_second': 0.571, 'total_flos': 1.857035730640896e+16, 'train_loss': 4.562057102260305, 'epoch': 1.0})